# Cvičení 2: Detekce slunečnic

Cílem cvičení je detekce slunečnic na fotografiích z pole. Květy najdeme detektorem blobů ve škálovém prostoru (např. **Laplacian of Gaussian, LoG**) a přebytečné bloby odfiltrujeme podle barvy okolí. Obrázky jsou v souborech `data/sunflowers{1-4}.jpg`.

1. Načtení obrázků a referenčních dat
2. Histogram odstínu vzorové slunečnice
3. Detekce květů
4. Vyhodnocení detekce
5. Bonus: robustnější vyhodnocení pomocí IoU

Obecné pokyny:
- Můžete používat [OpenCV](https://docs.opencv.org/) nebo např. [scikit-image](https://scikit-image.org/).
- Výsledky vykreslujte pomocí [matplotlib](https://matplotlib.org/).
- Svůj kód pište na místa označená komentářem `# TODO: implement`.
- Každá sekce obsahuje vzorový výstup. Váš výsledek nemusí být na pixel stejný, ale měl by vypadat podobně.
- Za každým bodem je buňka s testy ze souboru `tests/test_sunflowers.py`.
- K obrázkům patří **referenční anotace** (ground truth) ve formátu YOLO: každý řádek souboru `data/sunflowers{i}.txt` popisuje jednu slunečnici ve tvaru `třída cx cy w h`, kde `(cx, cy)` je střed obdélníku a `(w, h)` jeho šířka a výška, vše normalizované na rozsah 0 až 1 vzhledem k rozměrům obrázku.

In [ ]:
import sys
from collections.abc import Sequence

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
import scipy.stats
from skimage

if ".." not in sys.path:
    sys.path.append("..")
from tests import test_sunflowers

## 1. Načtení obrázků a referenčních dat

- Načtěte obrázky `../data/sunflowers{1-4}.jpg`, převeďte je do RGB a uložte do seznamu `images`.
- Implementujte funkci `load_yolo_boxes(path, shape)`, která načte anotace ve formátu YOLO a vrátí pole tvaru `(N, 4)` s obdélníky `(x1, y1, x2, y2)` v **pixelech**.
  - `shape` je rozměr obrázku `(výška, šířka, ...)`, podle kterého se normalizované souřadnice přepočítají na pixely.
  - Sloupec s třídou ignorujte, všechny anotované objekty jsou slunečnice.
- Anotace načtěte pro všechny čtyři obrázky do seznamu `boxes_true`. Seznamy `images` a `boxes_true` používají i testy detekce v bodě 4.
- Vypište počet anotovaných slunečnic v každém obrázku a obrázky vykreslete vedle sebe s anotovanými obdélníky.

Vzorový výstup:

<figure class="image">
  <img src="../figures/sunflowers-expected_outputs_load.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
def load_yolo_boxes(path: str, shape: tuple[int, ...]) -> NDArray[np.float64]:
    """
    Load boxes in the YOLO format and convert them to pixel coordinates.

    Args:
        path: Path to the text file with lines `class cx cy w h` (normalized coordinates).
        shape: Shape of the image (height, width, ...).
    Returns:
        Array of shape (N, 4) with boxes (x1, y1, x2, y2) in pixels.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement

# Nacteni obrazku a anotaci
# images = ...
# boxes_true = ...

# Vykresleni
# ...

########################################

In [ ]:
test_sunflowers.TestLoadData.eval(images=images, boxes_true=boxes_true, load_boxes_fn=load_yolo_boxes)

## 2. Histogram odstínu vzorové slunečnice

Květ poznáme podle barvy: žluté okvětní lístky kolem hnědého středu mají charakteristické rozdělení odstínů (hue). Jako vzor použijeme obrázek slunečnice `../data/sunflower_template.jpg` a později s ním budeme porovnávat výřezy z testovacích obrázků.

- Implementujte funkci `hue_histogram(rgb)`, která z obrázku (nebo výřezu) spočítá histogram **pouze složky H** v prostoru HSV.
  - Odstín převeďte na rozsah 0 až 1 (kanál H má v OpenCV pro 8bitové obrázky rozsah 0 až 179, ve scikit-image už je v rozsahu 0 až 1).
  - Histogram pokrývá celý rozsah odstínu 0 až 1 stejně širokými biny. Počet binů zvolte sami, doporučeno je 100.
  - Histogram **normalizujte na jednotkový součet** (součet hodnot je 1), aby šlo o rozdělení pravděpodobnosti odstínu. Výřezy různé velikosti se tak dají porovnávat a Kullback-Leiblerova divergence je definována právě pro rozdělení pravděpodobnosti.
- Načtěte obrázek `../data/sunflower_template.jpg`, převeďte jej do RGB (`template_rgb`), spočítejte jeho histogram odstínu a uložte jej do proměnné `template_hist`.
- Šablonu a její histogram vykreslete.
- Test ověří funkci na malém umělém obrázku složeném z pixelů několika barev se známým odstínem. Projde pro libovolný počet binů (alespoň 10).

Vzorový výstup:

<figure class="image">
  <img src="../figures/sunflowers-expected_outputs_template.png" alt="Vzorový výstup" style="width: 9.6in"/>
</figure>

In [ ]:
def hue_histogram(rgb: NDArray[np.uint8], bins: int = 100) -> NDArray[np.float64]:
    """
    Compute the histogram of the hue channel.

    Args:
        rgb: Image or crop in RGB order of dtype uint8.
        bins: Number of bins.
    Returns:
        Histogram with `bins` bins on the hue range 0..1 normalized to the unit sum.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement

# template_hist = ...

########################################

In [ ]:
test_sunflowers.TestHueHistogram.eval(hist_fn=hue_histogram)

## 3. Detekce květů

Střed květu je tmavý blob na světlých žlutých okvětních lístcích. Bloby různé velikosti najde detektor ve škálovém prostoru, který pro každý blob vrací střed a měřítko $\sigma$. Blobů bude hodně: nejen středy květů, ale i tmavé stíny mezi listy. Slunečnice mezi nimi vybereme podle barvy okolí.

- Implementujte funkci `detect_sunflowers(rgb)`, která najde slunečnice v obrázku a vrátí je jako **seznam blobů** `(x, y, sigma)`: střed v pixelech a měřítko $\sigma$. Počet květů je pak délka seznamu.
- Postup je na vás, doporučujeme následující kroky:
  1. **Detekce blobů.** Doporučujeme detektor **Laplacian of Gaussian (LoG)** [`skimage.feature.blob_log`](https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.blob_log) na šedotónovém obrázku s hodnotami 0 až 1.
     - LoG hledá **světlé** bloby, středy květů jsou tmavé. Použijte proto negativ obrazu $1 - \text{gray}$.
     - Nastavte rozsah měřítek (`min_sigma`, `max_sigma`, `num_sigma`) a `log_scale=True`. Vhodný rozsah je zhruba $\sigma \in [5, 90]$.
     - Pozor na pořadí souřadnic: `blob_log` vrací řádky `(y, x, sigma)`, kdežto `detect_sunflowers` vrací `(x, y, sigma)`.
     - Použít lze i jiný detektor, např. `blob_dog` nebo `blob_doh` ze scikit-image nebo `cv.SimpleBlobDetector` z OpenCV. Vrací-li detektor místo měřítka poloměr blobu $r$ (nebo průměr $2r$), převeďte jej na měřítko $\sigma \approx r / \sqrt{2}$.
  2. **Výřez okolí.** Pro každý blob vezměte čtvercový výřez o straně přibližně $4\sigma$ (tedy $\pm 2\sigma$ od středu). Bloby, jejichž výřez zasahuje mimo obrázek, vynechte.
  3. **Porovnání se šablonou.** Z výřezu spočítejte histogram odstínu `hue_histogram` a porovnejte jej s histogramem šablony `template_hist` pomocí **Kullback-Leiblerovy divergence**
     $$
     D_\mathrm{KL}(P \parallel Q) = \sum_i P_i \log\frac{P_i}{Q_i},
     $$
     kterou spočítá `scipy.stats.entropy(P, Q)`. Pokud by v histogramech byly nuly, divergence vyjde nekonečná nebo `NaN`. Přičtěte proto k oběma histogramům malé číslo, např. $0{,}001$.
  4. **Výběr slunečnic.** Bloby s divergencí **menší než zvolený práh** (zkuste hodnoty kolem 0,5 až 1,5) ponechte. Čím nižší práh, tím méně falešných detekcí, ale i méně nalezených květů.
- Detekci spusťte na všech obrázcích a výsledky uložte do seznamu `detections` (pro každý obrázek seznam blobů `(x, y, sigma)`). Vypište počty nalezených a anotovaných slunečnic a výsledky vykreslete, např. jako kružnice o poloměru $\sqrt{2}\,\sigma$.
- Test ověří funkci na umělém obrázku trávy a nebe, do kterého jsou vloženy tmavé kotouče a zmenšená šablona slunečnice ve třech velikostech. Funkce musí najít každou slunečnici (blob blízko středu květu s přiměřeným měřítkem $\sigma$) a nesmí najít žádný z kotoučů: jsou to bloby, ale ne slunečnice.
- **Vyhodnocení podle pozic:** kvalitu detekce na fotografiích ověří testy v bodě 4. Vyhodnocují vaše bloby `detections` vůči anotacím `boxes_true` **vaší funkcí** `evaluate_blobs`, kterou naprogramujete v bodě 4. Detekovaný blob je **správná detekce** (TP), pokud jeho střed leží uvnitř některého anotovaného obdélníku. Že je jeden obdélník nalezen vícekrát, se neřeší. Bloby mimo všechny obdélníky jsou **falešné** (FP) a obdélníky bez jediné správné detekce jsou **přehlédnuté** (FN).
  - Z TP, FP a FN se spočítá zjednodušené skóre $F_1$ pro každý obrázek zvlášť. Požadované hodnoty jsou uvedeny v bodě 4.
- Který z obrázků je nejtěžší a proč? Jaké chyby metoda dělá (falešné detekce, přehlédnuté květy) a čím by šlo metodu vylepšit?

Vzorový výstup:

<figure class="image">
  <img src="../figures/sunflowers-expected_outputs_detection.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
def detect_sunflowers(rgb: NDArray[np.uint8]) -> list[tuple[float, float, float]]:
    """
    Detect sunflowers as dark blobs whose surroundings have a hue histogram similar to the template.

    Args:
        rgb: Image in RGB order of dtype uint8.
    Returns:
        List of blobs (x, y, sigma) in pixels, one per detected sunflower.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement

# detections = ...

########################################

In [ ]:
test_sunflowers.TestDetectSunflowers.eval(detect_fn=detect_sunflowers)

## 4. Vyhodnocení detekce

Aby šlo kvalitu detekce posoudit, potřebujeme ji **změřit**. Naprogramujte vyhodnocovací funkci a použijte ji i pro vlastní ladění (např. výběr prahu divergence).

- Implementujte funkci `evaluate_blobs(blobs, boxes_true)`, která porovná detekované bloby `(x, y, sigma)` s anotovanými obdélníky `(x1, y1, x2, y2)` a vrátí trojici `(TP, FP, FN)`:
  - **TP** (true positives): počet blobů, jejichž střed `(x, y)` leží uvnitř **některého** anotovaného obdélníku (včetně hranice). Jeden obdélník může být zasažen i vícekrát, každý takový blob se počítá jako TP.
  - **FP** (false positives): počet blobů, které nelze k žádnému obdélníku přiřadit, tedy leží mimo všechny obdélníky.
  - **FN** (false negatives): počet anotovaných obdélníků, které nezasáhl žádný blob.
  - Funkce musí zvládnout i prázdný seznam blobů.
- Funkci ověří test na vymyšlených datech: tři obdélníky a několik sad detekcí (bezchybná detekce, posunuté bloby, přehlédnutý květ, falešné bloby, prázdný seznam, malý blob uvnitř obdélníku a duplicitní zásahy).
- Funkci použijte na výsledky detekce ze 3. bodu: pro každý obrázek vypište TP, FP, FN a skóre $F_1$.
- Testy detekce `TestDetectImage1` až `TestDetectImage4` vyhodnocují vaše výsledky `detections` vůči anotacím `boxes_true` **vaší** funkcí `evaluate_blobs`, a proto musí nejprve projít její test. Požadované skóre $F_1 = \dfrac{2\,\mathrm{TP}}{2\,\mathrm{TP} + \mathrm{FP} + \mathrm{FN}}$ detekce je pro jednotlivé obrázky alespoň **0,55**, **0,55**, **0,7** a **0,75**.

Vzorový výstup:

<figure class="image">
  <img src="../figures/sunflowers-expected_outputs_evaluation.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
def evaluate_blobs(blobs: Sequence[Sequence[float]], boxes_true: Sequence[Sequence[float]]) -> tuple[int, int, int]:
    """
    Evaluate detected blobs against the annotated boxes. A blob is a hit if its center is inside some box.

    Args:
        blobs: Detected blobs (x, y, sigma) in pixels.
        boxes_true: Annotated boxes (x1, y1, x2, y2) in pixels.
    Returns:
        Number of true positives (blobs inside some box), false positives (blobs outside of all boxes)
        and false negatives (boxes without any blob inside).
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
test_sunflowers.TestEvaluateBlobs.eval(evaluate_fn=evaluate_blobs)

In [ ]:
########################################
# TODO: implement



########################################

In [ ]:
test_sunflowers.TestDetectImage1.eval(detections=detections, boxes_true=boxes_true, evaluate_fn=evaluate_blobs)

In [ ]:
test_sunflowers.TestDetectImage2.eval(detections=detections, boxes_true=boxes_true, evaluate_fn=evaluate_blobs)

In [ ]:
test_sunflowers.TestDetectImage3.eval(detections=detections, boxes_true=boxes_true, evaluate_fn=evaluate_blobs)

In [ ]:
test_sunflowers.TestDetectImage4.eval(detections=detections, boxes_true=boxes_true, evaluate_fn=evaluate_blobs)

## 5. Bonus: robustnější vyhodnocení pomocí IoU

Vyhodnocení podle středů je shovívavé: neřeší velikost blobu a jeden květ může být nalezen vícekrát (a vícekrát se za to započítá TP). Standardní vyhodnocení detekce porovnává obdélníky pomocí **Jaccardova indexu** (IoU, intersection over union)
$$
\mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$
kde $A$ a $B$ jsou plochy dvou obdélníků, a každou anotaci může zasáhnout nejvýše jedna detekce.

- Implementujte funkci `box_iou(a, b)`, která spočítá IoU dvou obdélníků `(x1, y1, x2, y2)`.
- Implementujte funkci `evaluate_blobs_iou(blobs, boxes_true, iou_threshold=0.5)` se stejným rozhraním jako `evaluate_blobs` (vrací `(TP, FP, FN)`). Každý blob `(x, y, sigma)` nahraďte čtvercem o straně $4\sigma$ se středem v `(x, y)`, tedy stejným výřezem, jaký doporučujeme v bodě 3.
  - Detekce procházejte jednu po druhé a každou přiřaďte k dosud **nepřiřazené** anotaci s největším IoU. Je-li IoU alespoň `iou_threshold`, jde o správnou detekci (TP) a anotace se označí jako přiřazená. Jedna anotace tak může být přiřazena nejvýše jedné detekci.
  - Zbylé detekce jsou falešné (FP), nepřiřazené anotace jsou přehlédnuté (FN).
- Funkci `box_iou` ověří test na jednoduchých dvojicích obdélníků (shodné, disjunktní, dotýkající se, částečný překryv, vnořené).
- Funkci `evaluate_blobs_iou` ověří test na stejných vymyšlených datech jako `evaluate_blobs` (společné případy musí dát stejný výsledek) a na případech, kde se obě vyhodnocení liší (malý blob uvnitř obdélníku, duplicitní zásahy).
- Funkci použijte na výsledky detekce, vypište přesnost (precision) $\mathrm{TP} / (\mathrm{TP} + \mathrm{FP})$ a úplnost (recall) $\mathrm{TP} / (\mathrm{TP} + \mathrm{FN})$ pro každý obrázek a porovnejte je s vyhodnocením podle středů.
- Požadavky testu detekce (vyhodnocuje vaše `detections` vůči `boxes_true` vaší funkcí `evaluate_blobs_iou`): přesnost i úplnost vypočtená přes všechny obrázky dohromady alespoň **0,6** (při IoU 0,5).
- Co ovlivňuje přesnost a co úplnost? Jak je změní zvolený práh divergence?

Vzorový výstup:

<figure class="image">
  <img src="../figures/sunflowers-expected_outputs_bonus.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """
    Compute the intersection over union of two boxes.

    Args:
        a: First box (x1, y1, x2, y2) in pixels.
        b: Second box (x1, y1, x2, y2) in pixels.
    Returns:
        IoU in the range 0..1.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
def evaluate_blobs_iou(
    blobs: Sequence[Sequence[float]],
    boxes_true: Sequence[Sequence[float]],
    iou_threshold: float = 0.5,
) -> tuple[int, int, int]:
    """
    Evaluate detected blobs against the annotated boxes by IoU with unique assignment of the boxes.

    Args:
        blobs: Detected blobs (x, y, sigma) in pixels. Each blob is represented by the square with the side 4 * sigma.
        boxes_true: Annotated boxes (x1, y1, x2, y2) in pixels.
        iou_threshold: Minimal IoU of a detection and an annotation for the detection to be correct.
    Returns:
        Number of true positives (detections matched to a different annotation each), false positives
        (the other detections) and false negatives (annotations without a detection).
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
########################################
# TODO: implement



########################################

In [ ]:
test_sunflowers.TestBoxIoU.eval(box_iou_fn=box_iou)

In [ ]:
test_sunflowers.TestEvaluateBlobsIoU.eval(evaluate_fn=evaluate_blobs_iou)

In [ ]:
test_sunflowers.TestDetectionIoU.eval(detections=detections, boxes_true=boxes_true, evaluate_fn=evaluate_blobs_iou)